# Qwen3-4B: линкер владельцев на наших синтетических сессиях

В Colab выберите **Runtime → Change runtime type → GPU**. Модель загружается
с Hugging Face и работает на GPU Colab, без Radeon API и ключа Gemma.
По умолчанию Qwen3-4B, 4-bit NF4, reasoning выключен. Можно выбрать Qwen3-1.7B.
Доступность GPU и точное потребление памяти проверяются при запуске.

Загрузите подготовленный `qwen-linker-colab-bundle.zip` в ячейке ниже.
Вход содержит тексты и эталонные спаны, **без владельцев**; эталон используется
только оценщиком. Это оценка линкера, а не полного NER-сервиса.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'accelerate>=1.2,<2', 'bitsandbytes>=0.48,<0.49',
    'pydantic>=2.10,<3', 'sentencepiece>=0.2,<0.3'])


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, hashlib, json
uploaded = files.upload()  # Выберите qwen-linker-colab-bundle.zip
archive = next(Path(name) for name in uploaded if name.endswith('.zip'))
ROOT = Path('/content/session-pii-colab')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (ROOT / name).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError('Недопустимый путь внутри архива')
    z.extractall(ROOT)
os.chdir(ROOT)
manifest = json.loads((ROOT/'bundle-manifest.json').read_text())
for name, digest in manifest['files'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == digest, name
print('Проверено файлов:', len(manifest['files']))
print('Сессий:', manifest['sessions'])


## Где сохранять результаты

Для сохранения после отключения Colab поставьте `USE_DRIVE=True` и разрешите
подключение своего Drive. Иначе файлы остаются только во временном `/content`;
скачивайте их последней ячейкой до завершения runtime. При продолжении используйте
прежнюю папку. Для другой модели или параметров задайте новое имя запуска.


In [ ]:
USE_DRIVE = False
MODEL_ID = 'Qwen/Qwen3-4B'  # Альтернатива: Qwen/Qwen3-1.7B
RUN_NAME = MODEL_ID.split('/')[-1] + '-nf4-no-thinking-owner-v2'
MAX_NEW_TOKENS = 1024
MAX_INPUT_TOKENS = 8192
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/session-pii-linker') / RUN_NAME
else:
    OUTPUT = ROOT / 'results' / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Результаты:', OUTPUT)


In [ ]:
import torch, importlib.metadata
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
assert torch.cuda.is_available(), 'Включите GPU в настройках Colab'
print(torch.cuda.get_device_name(0))
saved = OUTPUT / 'manifest.json'
old = json.loads(saved.read_text())['model'] if saved.exists() else None
if old and old['model_id'] != MODEL_ID:
    raise ValueError('Для другой модели выберите новую папку результатов')
REVISION = old['revision'] if old else HfApi().model_info(MODEL_ID).sha
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,
    trust_remote_code=False, device_map={'': 0}, torch_dtype=compute_dtype,
    attn_implementation='sdpa', quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype))
model.eval()
sys.path[:0] = [str(ROOT/'tools'), str(ROOT/'src'), str(ROOT)]
from colab_linker import LocalCompletion, run_sessions, evaluate, write, LOCAL_OWNER_GUIDANCE
completion = LocalCompletion(model, tokenizer, MAX_INPUT_TOKENS)
identity = dict(model_id=MODEL_ID, revision=REVISION, quantization='nf4-double',
    compute_dtype=str(compute_dtype), thinking=False, do_sample=False,
    max_new_tokens=MAX_NEW_TOKENS, max_input_tokens=MAX_INPUT_TOKENS,
    local_guidance_sha256=hashlib.sha256(LOCAL_OWNER_GUIDANCE.encode()).hexdigest(),
    schema_mode='schema in prompt; strict local validation; no constrained decoding',
    bundle_sha256=hashlib.sha256((ROOT/'bundle-manifest.json').read_bytes()).hexdigest())
write(OUTPUT/'environment.json', dict(gpu=torch.cuda.get_device_name(0), identity=identity,
    versions={p:importlib.metadata.version(p) for p in
              ['torch','transformers','accelerate','bitsandbytes','pydantic']}))
print('Revision:', REVISION)
print('Выделено GPU, GiB:', round(torch.cuda.memory_allocated()/2**30, 2))


## Пробный запуск

Первые сессии выполняются с сохранением всех запросов и ответов. В полном запуске
успешно завершённые сессии повторяться не будут. Неуспешные тоже не повторяются
автоматически; для явного повтора используйте `retry_failed=True`.
JSON не исправляется автоматически: обрезанный или неверный ответ считается ошибкой.


In [ ]:
state = run_sessions(ROOT/'data/inference.json', OUTPUT, completion, identity, limit=5)
summary = evaluate(OUTPUT, ROOT/'data/reference.json', ROOT/'data/gemma-baseline.json')
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Посмотрите сырой ответ до запуска остальных сессий.
paths = sorted(OUTPUT.glob('*/attempt-*/call-*-response.json'))
if paths:
    raw = json.loads(paths[0].read_text())
    print('Первый ответ:', raw['choices'][0]['message']['content'])
print('Совпавших групп:', summary['quality']['exact_owner_groups'], '/', summary['completed'])
print('complete означает успешный контракт, а не правильное назначение владельца.')


## Полный прогон и продолжение

Выполните ячейку после проверки пробных ответов. При остановке запустите её снова:
сохраняется каждая завершённая сессия. После перезапуска runtime сначала повторите
установку, загрузку архива и модели с прежним OUTPUT. Незавершённая сессия начнётся
заново, старая попытка сохранится. Веса модели в архив результатов не включаются.


In [ ]:
try:
    state = run_sessions(ROOT/'data/inference.json', OUTPUT, completion, identity)
finally:
    summary = evaluate(OUTPUT, ROOT/'data/reference.json', ROOT/'data/gemma-baseline.json')
    print(json.dumps(summary, ensure_ascii=False, indent=2))


Оценщик показывает покрытие и ошибки отдельно от качества успешных ответов:
pairwise Precision/Recall/F1, точные группы владельцев, Precision/Recall и два F2
алертов. В `common_subset` — сравнение с Gemma на одинаковых успешных сессиях.
Неизвестные владельцы исключены из pairwise, но учтены в точных группах.

Локальное время включает токенизацию и генерацию, исключает загрузку весов;
время генерации дополнительно записано после CUDA synchronization. Это не RPS-тест.
NF4 и отсутствие constrained decoding отличают запуск от серверного Gemma.
Синтетические короткие сценарии не заменяют независимый тест реальных чатов.


In [ ]:
import shutil
archive = shutil.make_archive('/content/' + RUN_NAME + '-results', 'zip', OUTPUT)
files.download(archive)
